In [12]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import phik
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, ConfusionMatrixDisplay,)
from sklearn.calibration import calibration_curve
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.trial import FixedTrial
import math


load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="hotel")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM hotels", engine)

print(df.shape)
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df['is_canceled'].mean())

(119390, 32)
(87396, 32)
0.27489816467572886


In [13]:
df["total_night"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df = df.drop(columns=["company","assigned_room_type","distribution_channel", "reservation_status", "reservation_status_date", "stays_in_weekend_nights", "stays_in_week_nights", "arrival_date_year", "arrival_date_month"])
df['country'] = df['country'].fillna("PRT")
df['agent'] = df['agent'].fillna(0)
df = df[df['lead_time'] < 600]
df = df[(df['adr'] < 5000) & (df['adr'] >= 0)]
df["adr_zero"] = (df["adr"] == 0).astype(int)
df = df[df['total_night'] < 31]

In [21]:
x = df.drop(columns='is_canceled')
y = df['is_canceled']

x_tmp, x_test, y_tmp, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_tmp.shape, y_tmp.shape)
x_tr, x_val, y_tr, y_val = train_test_split(x_tmp, y_tmp, test_size=0.25, random_state=42)


(69885, 24) (69885,)


60% train, 20% val, 20% test